In [2]:
import sys
import os
# only go up a folder if we are currently inside the notebooks folder
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    
project_root = os.getcwd() 
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [3]:
%load_ext autoreload
%autoreload 2

## Phase 1: Verification
Verification if all files are imported successfully.

In [4]:
import pandas as pd

In [5]:
reviews_df = pd.read_csv("csv/tokopedia_product_reviews_2025.csv")
reviews_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 65543 entries, 0 to 65542
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   review_text       65543 non-null  str  
 1   review_date       65543 non-null  str  
 2   review_id         65543 non-null  int64
 3   product_name      65543 non-null  str  
 4   product_category  65543 non-null  str  
 5   product_variant   26749 non-null  str  
 6   product_price     65543 non-null  int64
 7   product_url       65543 non-null  str  
 8   product_id        65543 non-null  int64
 9   rating            65543 non-null  int64
 10  sold_count        65543 non-null  int64
 11  shop_id           65543 non-null  int64
 12  sentiment_label   65543 non-null  str  
dtypes: int64(6), str(7)
memory usage: 6.5 MB


65543 rows and 13 columns, column names seems correct as well. product_variant has missing values, but doesn't matter for now.

In [6]:
reviews_df.head(5)

,review_text,review_date,review_id,product_name,product_category,product_variant,product_price,product_url,product_id,rating,sold_count,shop_id,sentiment_label
0,baru sekali ini terima brg dr belanja online d...,2024-12-22,1134256160,Telur Ayam Kampung Asli - Telur Mengandung Ome...,Makanan & Minuman,Box Polos,87000,https://www.tokopedia.com/indofarmproduct/telu...,4601033481,5,1000000,8672687,positive
1,cocok bgt aku sama telur nya. nga Amis menurut...,2025-02-25,1242584634,Telur Ayam Kampung Asli - Telur Mengandung Ome...,Makanan & Minuman,Box Polos,87000,https://www.tokopedia.com/indofarmproduct/telu...,4601033481,5,1000000,8672687,positive
2,Telornya sudah sampai di rumah dengan kemasan ...,2025-07-15,1573444677,Telur Ayam Kampung Asli - Telur Mengandung Ome...,Makanan & Minuman,Box Polos,87000,https://www.tokopedia.com/indofarmproduct/telu...,4601033481,5,1000000,8672687,positive
3,Telor sudah diterima dengan baik dan tidak ada...,2025-07-20,1581728541,Telur Ayam Kampung Asli - Telur Mengandung Ome...,Makanan & Minuman,Box Polos,87000,https://www.tokopedia.com/indofarmproduct/telu...,4601033481,5,1000000,8672687,positive
4,"Alhamdulillah penjual amanah,Telor nya terbaik...",2023-04-24,881041355,Telur Ayam Kampung Asli - Telur Mengandung Ome...,Makanan & Minuman,Box Full Design,87000,https://www.tokopedia.com/indofarmproduct/telu...,4601033481,5,1000000,8672687,positive


In [7]:
cat_neg_reviews_df = reviews_df[["product_category", "sentiment_label"]].copy()
cat_neg_reviews_df

,product_category,sentiment_label
0,Makanan & Minuman,positive
1,Makanan & Minuman,positive
2,Makanan & Minuman,positive
3,Makanan & Minuman,positive
4,Makanan & Minuman,positive
...,...,...
65538,Olahraga,positive
65539,Olahraga,positive
65540,Olahraga,positive
65541,Olahraga,positive


In [8]:
cat_neg_reviews_df = cat_neg_reviews_df[cat_neg_reviews_df["sentiment_label"] == "negative"]
cat_neg_reviews_df

,product_category,sentiment_label
12,Makanan & Minuman,negative
49,Makanan & Minuman,negative
51,Makanan & Minuman,negative
54,Makanan & Minuman,negative
77,Makanan & Minuman,negative
...,...,...
64106,Olahraga,negative
64133,Olahraga,negative
64668,Olahraga,negative
64945,Olahraga,negative


In [9]:
cat_neg_reviews_df = cat_neg_reviews_df.groupby("product_category").agg(
    total_negatives=("sentiment_label", "size")
).reset_index()
cat_neg_reviews_df

,product_category,total_negatives
0,Elektronik,24
1,Handphone & Tablet,62
2,Kesehatan,39
3,Makanan & Minuman,263
4,Olahraga,277
5,Pertukangan,133


In [10]:
cat_tot_reviews_df = reviews_df[["product_category", "review_text"]].copy()
cat_tot_reviews_df

,product_category,review_text
0,Makanan & Minuman,baru sekali ini terima brg dr belanja online d...
1,Makanan & Minuman,cocok bgt aku sama telur nya. nga Amis menurut...
2,Makanan & Minuman,Telornya sudah sampai di rumah dengan kemasan ...
3,Makanan & Minuman,Telor sudah diterima dengan baik dan tidak ada...
4,Makanan & Minuman,"Alhamdulillah penjual amanah,Telor nya terbaik..."
...,...,...
65538,Olahraga,"kwalitas bagus, pokoknya rekomendit deh"
65539,Olahraga,Sesuai harga
65540,Olahraga,Sesuai harga
65541,Olahraga,Mantap


In [11]:
cat_tot_reviews_df = cat_tot_reviews_df.groupby("product_category").agg(
    total_reviews=("review_text", "size")
).reset_index()
cat_tot_reviews_df

,product_category,total_reviews
0,Elektronik,4202
1,Handphone & Tablet,7423
2,Kesehatan,8959
3,Makanan & Minuman,17859
4,Olahraga,15600
5,Pertukangan,11500


In [12]:
trueneg_df = pd.merge(cat_neg_reviews_df, cat_tot_reviews_df, on="product_category", how="inner")
trueneg_df

,product_category,total_negatives,total_reviews
0,Elektronik,24,4202
1,Handphone & Tablet,62,7423
2,Kesehatan,39,8959
3,Makanan & Minuman,263,17859
4,Olahraga,277,15600
5,Pertukangan,133,11500


In [13]:
trueneg_df["true_negative"] = (trueneg_df["total_negatives"] / trueneg_df["total_reviews"]) * 100
trueneg_df

,product_category,total_negatives,total_reviews,true_negative
0,Elektronik,24,4202,0.571157
1,Handphone & Tablet,62,7423,0.835242
2,Kesehatan,39,8959,0.435316
3,Makanan & Minuman,263,17859,1.472647
4,Olahraga,277,15600,1.775641
5,Pertukangan,133,11500,1.156522


In [14]:
trueneg_df_tidy = trueneg_df[["product_category", "true_negative"]].to_dict(orient="list")
trueneg_df_tidy

{'product_category': ['Elektronik',
  'Handphone & Tablet',
  'Kesehatan',
  'Makanan & Minuman',
  'Olahraga',
  'Pertukangan'],
 'true_negative': [0.5711565920990005,
  0.8352418159773677,
  0.43531644156713917,
  1.4726468447281482,
  1.7756410256410258,
  1.1565217391304348]}

In [15]:
for_color = pd.DataFrame(trueneg_df_tidy["product_category"])
for_color

,0
0,Elektronik
1,Handphone & Tablet
2,Kesehatan
3,Makanan & Minuman
4,Olahraga
5,Pertukangan


----------------


In [ ]:
bubble_df = reviews_df[["product_id", "product_name", "product_category", "product_price", "rating", "sold_count"]].copy()
bubble_df = bubble_df.groupby(["product_id", "product_name", "product_category", "product_price"]).agg(
    average_rating=("rating", "mean"),
    total_sold=("sold_count", "sum")
).reset_index()
bubble_df

,,,,average_rating,total_sold
product_id,product_name,product_category,product_price,,
4298375,Jual Teleskop Bintang Celestron PowerSeeker 60AZ,Olahraga,1768000,5.000000,2000
12313359,MD Organic Red Rice Pecah Kulit 2kg,Makanan & Minuman,78000,5.000000,60000
21172457,Blue Diamond Chalk - 2 Piece Billiard Chalk - Kapur Biliar - Biru,Olahraga,150000,4.950000,40000
22825312,Aramith Billiard Ball Cleaner - 8.4 Fl. Bottle - Pembersih Bola Biliar,Olahraga,160000,4.777778,4500
22845300,Aramith Billiard Training Ball - by Jim Rempe - Bola Latihan Biliar,Olahraga,750000,4.941176,1700
...,...,...,...,...,...
102656595877,Dux Ducis Case Compatible for Samsung Tab A11 Plus | Tab A11 | Tab A9 Plus | Tab A9 - Puff Cover Casing - Safe for Kids / Children,Handphone & Tablet,175000,4.909091,1100
102662280557,Dux Ducis Case Compatible for Samsung Tab A11 Plus | Tab A9 Plus - MAGI Cover Casing,Handphone & Tablet,185000,5.000000,550
102671438736,"Smiles & Tides Baby Rashguard & Swim Diapers, Baju Renang Anak & Bayi",Olahraga,269900,5.000000,500
